# TO-DO: BigQuery Connectivity


*   ~~Shift description columns to a completely different dataframe (e.g., company_id, description)~~ (Now "company_description_df")
*   ~~Create csv files from dataframes with dropped descriptions~~
*   ~~Upload new csv files to BigQuery~~
*   Establish conneciton between BigQuery selects and Python statements

# TO-DO: Data Preprocessing


*   Check employee_counts.csv "time_recorded" column to ensure the maximum recorded time has been selected for every unique company_id. (i.e., Make sure the most updated employee count has been captured for each unique company.)

#Importing Primary Libraries

In [ ]:
# Import statements
import pandas as pd

#Importing Data

In [ ]:
# Importing dataset from kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arshkon/linkedin-job-postings")

print("Path to dataset files:", path)

100%|██████████| 159M/159M [00:01<00:00, 164MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/arshkon/linkedin-job-postings/versions/13


#Data Accessibility Tests

In [ ]:
# Creating an ancillary paths dictionary for the dataset
ancillary_directories = ['companies', 'jobs', 'mappings']
job_posts_path_dict = {}

for directories in ancillary_directories:
  job_posts_path_dict[directories] = path + '/' + directories

In [ ]:
# Grabbing the filename of the job postings data
job_posts_filename = path + '/' + 'postings.csv'

In [ ]:
# Creating a pandas dataframe from 'postings.csv'
job_posts_df = pd.read_csv(job_posts_filename)


In [ ]:
# TODO - Convert these dataframes into dictionaries (e.g., companies_dict{...}, jobs_dict{...}, etc.)

# Creating pandas dataframes for the ancillary directories
companies_df = pd.read_csv(job_posts_path_dict[ancillary_directories[0]] + '/companies.csv')
companies_industries_df = pd.read_csv(job_posts_path_dict[ancillary_directories[0]] + '/company_industries.csv')
companies_specialities_df = pd.read_csv(job_posts_path_dict[ancillary_directories[0]] + '/company_specialities.csv')
companies_employee_counts_df = pd.read_csv(job_posts_path_dict[ancillary_directories[0]] + '/employee_counts.csv')

#jobs_df = TODO - Jobs has several sub files
jobs_benefits_df = pd.read_csv(job_posts_path_dict[ancillary_directories[1]] + '/benefits.csv')
jobs_industries_df = pd.read_csv(job_posts_path_dict[ancillary_directories[1]] + '/job_industries.csv')
jobs_skills_df = pd.read_csv(job_posts_path_dict[ancillary_directories[1]] + '/job_skills.csv')
jobs_salaries_df = pd.read_csv(job_posts_path_dict[ancillary_directories[1]] + '/salaries.csv')

#mappings_df = TODO - Mappings has sub files
mappings_industries_df = pd.read_csv(job_posts_path_dict[ancillary_directories[2]] + '/industries.csv')
mappings_skills_df = pd.read_csv(job_posts_path_dict[ancillary_directories[2]] + '/skills.csv')

ancillary_datasets_list = [
    companies_df,
    companies_industries_df,
    companies_specialities_df,
    companies_employee_counts_df,
    jobs_benefits_df,
    jobs_industries_df,
    jobs_skills_df,
    jobs_salaries_df,
    mappings_industries_df,
    mappings_skills_df
]

# Creating Description Dataframe from Company Info

In [ ]:
# At some point, we may need to be very careful about maintaining our job posting IDs.
company_description_df = ancillary_datasets_list[0][['company_id', 'description']].copy()

In [ ]:
company_description_df.head()

,company_id,description
0,1009,"At IBM, we do more than work. We create. We cr..."
1,1016,Every day millions of people feel the impact o...
2,1025,Official LinkedIn of Hewlett Packard Enterpris...
3,1028,We’re a cloud technology company that provides...
4,1033,Accenture is a leading global professional ser...


In [ ]:
ancillary_datasets_list[0].drop(["description"], axis=1, inplace=True)

In [ ]:
# Dropping "address" (i.e., the street address) column of the company dataset as city, country, and zip code should be good enough
ancillary_datasets_list[0].drop(["address"], axis=1, inplace=True)

# CURRENT FILTERS: description, address

# Dataset Inspector

In [69]:
# Select dataset to inspect...
# 0 - companies/companies.csv
# 1 - companies/company_industries.csv
# 2 - companies/company_specialities.csv
# 3 - companies/employee_counts.csv
# ...
# 8 - mappings/industries.csv
# 9 - mappings/skills.csv

DATASET_INDEX = 0

current_dataset = ancillary_datasets_list[DATASET_INDEX]

In [70]:
# View information about current dataset
current_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24473 entries, 0 to 24472
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   company_id    24473 non-null  int64  
 1   name          24472 non-null  object 
 2   company_size  21699 non-null  float64
 3   state         24451 non-null  object 
 4   country       24473 non-null  object 
 5   city          24472 non-null  object 
 6   zip_code      24445 non-null  object 
 7   url           24473 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 1.5+ MB


In [71]:
# View first five rows of current dataset
current_dataset.head(10)

,company_id,name,company_size,state,country,city,zip_code,url
0,1009,IBM,7.0,NY,US,"Armonk, New York",10504,https://www.linkedin.com/company/ibm
1,1016,GE HealthCare,7.0,0,US,Chicago,0,https://www.linkedin.com/company/gehealthcare
2,1025,Hewlett Packard Enterprise,7.0,Texas,US,Houston,77389,https://www.linkedin.com/company/hewlett-packa...
3,1028,Oracle,7.0,Texas,US,Austin,78741,https://www.linkedin.com/company/oracle
4,1033,Accenture,7.0,0,IE,Dublin 2,0,https://www.linkedin.com/company/accenture
5,1035,Microsoft,7.0,Washington,US,Redmond,98052,https://www.linkedin.com/company/microsoft
6,1038,Deloitte,7.0,0,OO,Worldwide,0,https://www.linkedin.com/company/deloitte
7,1043,Siemens,7.0,0,DE,Munich,80333,https://www.linkedin.com/company/siemens
8,1044,PwC,7.0,0,GB,0,0,https://www.linkedin.com/company/pwc
9,1052,AT&T,7.0,TX,US,Dallas,75202,https://www.linkedin.com/company/att


In [ ]:
current_dataset.to_csv('filtered_companies.csv', index=False)

# Job Posts Manipulation

In [67]:
job_posts_df.head(10)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0
5,91700727,Downtown Raleigh Alliance,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,20.0,HOURLY,"Raleigh, NC",1481176.0,9.0,NaN,...,NaN,1.713456e+12,NaN,0,INTERNSHIP,USD,BASE_SALARY,35360.0,27601.0,37183.0
6,103254301,Raw Cereal,Producer,Company DescriptionRaw Cereal is a creative de...,300000.0,YEARLY,United States,81942316.0,7.0,NaN,...,NaN,1.712861e+12,NaN,0,CONTRACT,USD,BASE_SALARY,180000.0,NaN,NaN
7,112576855,NaN,Building Engineer,Summary: Due to the pending retirement of our ...,120000.0,YEARLY,"San Francisco, CA",NaN,2.0,NaN,...,NaN,1.712443e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,105000.0,94101.0,6075.0
8,1218575,Children's Nebraska,Respiratory Therapist,"At Children’s, the region’s only full-service ...",NaN,NaN,"Omaha, NE",721189.0,3.0,NaN,...,• Requires the ability to communicate effectiv...,1.712348e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,68102.0,31055.0
9,2264355,Bay West Church,Worship Leader,It is an exciting time to be a part of our chu...,NaN,MONTHLY,"Palm Bay, FL",28631247.0,5.0,350.0,...,"Knowledge, Skills and Abilities: 1. Proficient...",1.712456e+12,NaN,0,PART_TIME,USD,BASE_SALARY,4200.0,32905.0,12009.0


In [68]:
job_posts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [72]:
job_posts_descriptions_df = job_posts_df[["description", "company_name", "location", "skills_desc"]].copy()

In [73]:
job_posts_df.drop(["description", "company_name", "location", "skills_desc"], axis=1, inplace=True)

In [75]:
job_posts_df.to_csv('filtered_postings.csv', index=False)

# BigQuery + Notebook Connectivity

In [77]:
# CLOUD RUN TESTING
import requests
FUNCTION_URL = "https://query-job-postings-789520167511.us-east1.run.app"

response = requests.get(FUNCTION_URL)
print(response.status_code)
print(response.text)

200
Hello World!


In [ ]:
import requests
import pandas as pd

FUNCTION_URL = "TODO: Setup Google Cloud Function"

def bq_query(sql):
    response = requests.post(FUNCTION_URL, json={"query": sql})
    response.raise_for_status()
    return pd.DataFrame(response.json()["data"])

df = bq_query("SELECT * FROM linkedin_job_data.FIX_FIX_FIX WHERE condition = 'x'")